# CHUYÊN ĐỀ NLP: BÓC TÁCH, CHUẨN HÓA VÀ KHAI PHÁ DỮ LIỆU TUYỂN DỤNG IT
## DỰ ÁN: IT JOB MARKET INTELLIGENCE

### 1. Đặt Vấn Đề
Trong các sàn tuyển dụng IT (TopCV, TopDev, VietnamWorks), hơn 85% thông tin có giá trị nằm dưới dạng **văn bản phi cấu trúc (unstructured text)**:
- Tiêu đề công việc được đặt tự do theo sở thích nhà tuyển dụng.
- Mô tả công việc (JD), yêu cầu ứng viên, quyền lợi chứa lẫn mã HTML, ký tự đặc biệt, nhiều ngôn ngữ (Tiếng Việt, Tiếng Anh, Song ngữ).
- Mức lương và số năm kinh nghiệm được viết bằng câu chữ tự nhiên (`Up to 2,000 USD`, `2-3 năm kinh nghiệm`, `Lương thỏa thuận`).
- Các kỹ năng công nghệ nằm rải rác trong các đoạn văn dài.

Nếu không có **Xử lý Ngôn ngữ Tự nhiên (NLP - Natural Language Processing)**, hệ thống chỉ lưu trữ được các chuỗi văn bản thô, không thể thực hiện phân tích thống kê, lọc đa chiều, benchmark lương hay xây dựng biểu đồ ma trận kỹ năng.

### 2. Kiến Trúc Pipeline NLP 6 Giai Đoạn
NLP trong dự án đóng vai trò cầu nối chuyển hóa dữ liệu từ **Bronze Layer (Dữ liệu thô)** sang **Silver Layer (Dữ liệu sạch, quan hệ chuẩn hóa)** và **Gold Layer (Analytical Marts)**:

1. **Giai đoạn 1**: Tiền xử lý & Làm sạch văn bản HTML (Text Cleaning & Normalization).
2. **Giai đoạn 2**: Nhận diện ngôn ngữ bài đăng (Language Profiling: vi, en, vi-en).
3. **Giai đoạn 3**: Chuẩn hóa chức danh công việc (Job Title Normalization & Categorization).
4. **Giai đoạn 4**: Bóc tách thực thể số học (Salary & Experience Entity Extraction).
5. **Giai đoạn 5**: Trích xuất 76 kỹ năng IT chuẩn hóa (Skill Extraction / Regex-NER).
6. **Giai đoạn 6**: Phân loại tính chất yêu cầu (Required vs. Preferred Classification) & Ma trận đồng xuất hiện (Co-occurrence Matrix).


---
### PHẦN 1: CẤU HÌNH MÔI TRƯỜNG VÀ NẠP DỮ LIỆU THỰC TẾ TỪ TẦNG BRONZE

Chúng ta kết nối vào hệ thống trích xuất tầng Bronze để nạp toàn bộ dữ liệu tin tuyển dụng hợp lệ đã tích lũy từ các sàn TopCV, TopDev và VietnamWorks.


In [1]:
import os
import sys
import re
import hashlib
import json
import pandas as pd
import numpy as np

# Thiết lập đường dẫn gốc dự án
# Tự động tìm thư mục gốc dự án (chứa thư mục 'src')
def get_project_root():
    curr = os.path.abspath(os.getcwd())
    while curr != os.path.dirname(curr):
        if os.path.exists(os.path.join(curr, 'src')):
            return curr
        curr = os.path.dirname(curr)
    return os.path.abspath(os.path.join(os.getcwd(), '..', '..'))

PROJECT_ROOT = get_project_root()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.pipelines.silver.extractor import load_bronze_data
from src.pipelines.silver.transformers.skills import SKILLS_TAXONOMY
from src.pipelines.silver.transformers.job_skills import get_skill_patterns
from src.pipelines.silver.transformers.jobs import clean_job_title, parse_salary, parse_experience_years
from src.pipelines.silver.transformers.job_descriptions import clean_html_text, detect_language

# Nạp dữ liệu đồng bộ Bronze
bronze_data = load_bronze_data()
df = bronze_data["df_merged"].copy()

print(f"Tổng số bản ghi Bronze nạp vào bộ nhớ: {len(df):,} tin tuyển dụng")
print(f"Phân bố nguồn sàn:")
for s_name, s_id in [("TopCV", 1), ("TopDev", 2), ("VietnamWorks", 3)]:
    cnt = (df['source_id'] == s_id).sum()
    print(f"  - {s_name}: {cnt:,} tin ({cnt/len(df)*100:.1f}%)")


-> Nạp dữ liệu nguồn Bronze từ đĩa...
   Đã nạp thành công 2,752 bản ghi đồng bộ vào bộ nhớ.
Tổng số bản ghi Bronze nạp vào bộ nhớ: 2,752 tin tuyển dụng
Phân bố nguồn sàn:
  - TopCV: 1,507 tin (54.8%)
  - TopDev: 129 tin (4.7%)
  - VietnamWorks: 1,116 tin (40.6%)


---
### PHẦN 2: CÁC TRƯỜNG DỮ LIỆU VĂN BẢN ĐƯỢC SỬ DỤNG CHO NLP

Bảng tổng hợp các trường văn bản đầu vào từ tầng Bronze được đưa vào hệ thống NLP xử lý:

| Tên trường thô | Định dạng | Mô tả nội dung | Mục tiêu NLP |
| :--- | :--- | :--- | :--- |
| `title_raw` | Chuỗi văn bản | Tiêu đề tin do nhà tuyển dụng đặt tự do | Chuẩn hóa về nhóm chức danh chuẩn (title_normalized) |
| `description_raw` | HTML / Plain text | Bản mô tả chi tiết công việc | Làm sạch HTML, bóc tách kỹ năng, nhận diện ngôn ngữ |
| `requirements_raw` | HTML / Plain text | Yêu cầu chuyên môn, kỹ năng, kinh nghiệm | Bóc tách kỹ năng, phân loại Bắt buộc vs Ưu tiên |
| `benefits_raw` | HTML / Plain text | Quyền lợi, chế độ đãi ngộ, bảo hiểm | Làm sạch văn bản, phân tích văn hóa doanh nghiệp |
| `salary_raw` | Chuỗi văn bản | Mức lương hiển thị trên sàn | Bóc tách cận dưới, cận trên, đơn vị tiền, quy đổi VNĐ |
| `experience_raw` | Chuỗi văn bản | Yêu cầu số năm thâm niên | Bóc tách số năm min/max, phân vào nhóm kinh nghiệm |

Dưới đây là một ví dụ thực tế về một tin tuyển dụng ở trạng thái thô chưa qua xử lý:


In [2]:
# Xem mẫu một tin tuyển dụng thực tế chưa qua xử lý
sample_job = df.iloc[0]

print("=== MẪU DỮ LIỆU VĂN BẢN THÔ TRƯỚC KHI XỬ LÝ NLP ===")
print(f"1. ID việc làm        : {sample_job['job_id']}")
print(f"2. Tiêu đề thô (title) : {sample_job['title_raw']}")
print(f"3. Lương thô (salary) : {sample_job.get('salary_raw')}")
print(f"4. Kinh nghiệm thô    : {sample_job.get('experience_raw')}")
print(f"5. Đoạn đầu Mô tả thô :\n{str(sample_job.get('description_raw'))[:250]}...")
print(f"\n6. Đoạn đầu Yêu cầu thô:\n{str(sample_job.get('requirements_raw'))[:250]}...")


=== MẪU DỮ LIỆU VĂN BẢN THÔ TRƯỚC KHI XỬ LÝ NLP ===
1. ID việc làm        : 1
2. Tiêu đề thô (title) : Full-Stack Web Engineer Intern/Fresher  (AI Coding: PHP / Ruby / Python)
3. Lương thô (salary) : Thoả thuận
4. Kinh nghiệm thô    : Không yêu cầu
5. Đoạn đầu Mô tả thô :
None...

6. Đoạn đầu Yêu cầu thô:
REQUIREMENTS
Must have
Graduated (or final-year student) in Information Technology / Computer Science
Basic knowledge of web development: HTML, CSS, JavaScript, and concepts of MVC & OOP
English reading comprehension — able to self-search and learn f...


---
### PHẦN 3: TIỀN XỬ LÝ & LÀM SẠCH VĂN BẢN (TEXT CLEANING)

#### Quy trình xử lý của hàm `clean_html_text`:
1. **Giải mã thực thể HTML**: Chuyển các ký tự mã hóa như `&nbsp;`, `&amp;`, `&lt;`, `&gt;`, `&quot;` thành ký tự thông thường.
2. **Xử lý ngắt đoạn logic**: Thay thế các thẻ khối (`<br>`, `<p>`, `<li>`, `<div>`, `<h3>`) thành ký tự xuống dòng (`\n`) để giữ đúng cấu trúc đoạn văn bản.
3. **Loại bỏ thẻ rác**: Dùng biểu thức chính quy xóa toàn bộ các tag HTML còn lại.
4. **Chuẩn hóa đầu dòng (Bullet Points)**: Chuyển đổi các ký tự gạch đầu dòng lộn xộn (`*`, `+`, `-`, `•`) về chuẩn `- `.
5. **Khử khoảng trắng thừa**: Thu gọn khoảng trắng liên tiếp và loại bỏ các dòng trống rác.
6. **Mã băm toàn vẹn (SHA-256)**: Tạo trường `text_hash` để phát hiện bài đăng bị nhà tuyển dụng sửa đổi nội dung qua các đợt crawl.


In [3]:
# So sánh Before vs After trên một đoạn văn bản có chứa HTML phức tạp
dirty_html_sample = """
<div class="job-description">
  <h3>Mô tả công việc:</h3>
  <p>&bull;&nbsp;Tham gia phát triển hệ thống Backend bằng <strong>Python (Django / FastAPI)</strong>.</p>
  <p>&bull;&nbsp;Tối ưu hóa cơ sở dữ liệu PostgreSQL &amp; Redis cache.</p>
  <br>
  <h3>Yêu cầu ứng viên:</h3>
  <ul>
    <li>Có từ 2-3 năm kinh nghiệm lập trình Python.</li>
    <li>Ưu tiên ứng viên có kiến thức về Docker, Kubernetes và AWS cloud.</li>
  </ul>
</div>
"""

clean_sample = clean_html_text(dirty_html_sample)
sha256_hash = hashlib.sha256(clean_sample.encode('utf-8')).hexdigest()

print("--- [BEFORE]: VĂN BẢN CHỨA MÃ HTML RÁC ---")
print(dirty_html_sample.strip())

print("\n--- [AFTER]: VĂN BẢN ĐÃ LÀM SẠCH BẰNG NLP CLEANER ---")
print(clean_sample)

print(f"\nMã băm toàn vẹn nội dung (SHA-256 text_hash): {sha256_hash}")


--- [BEFORE]: VĂN BẢN CHỨA MÃ HTML RÁC ---
<div class="job-description">
  <h3>Mô tả công việc:</h3>
  <p>&bull;&nbsp;Tham gia phát triển hệ thống Backend bằng <strong>Python (Django / FastAPI)</strong>.</p>
  <p>&bull;&nbsp;Tối ưu hóa cơ sở dữ liệu PostgreSQL &amp; Redis cache.</p>
  <br>
  <h3>Yêu cầu ứng viên:</h3>
  <ul>
    <li>Có từ 2-3 năm kinh nghiệm lập trình Python.</li>
    <li>Ưu tiên ứng viên có kiến thức về Docker, Kubernetes và AWS cloud.</li>
  </ul>
</div>

--- [AFTER]: VĂN BẢN ĐÃ LÀM SẠCH BẰNG NLP CLEANER ---
Mô tả công việc:
&bull; Tham gia phát triển hệ thống Backend bằng Python (Django / FastAPI) .
&bull; Tối ưu hóa cơ sở dữ liệu PostgreSQL & Redis cache.
Yêu cầu ứng viên:
Có từ 2-3 năm kinh nghiệm lập trình Python.
Ưu tiên ứng viên có kiến thức về Docker, Kubernetes và AWS cloud.

Mã băm toàn vẹn nội dung (SHA-256 text_hash): 770e231ba683a227dc830f04fdc082a28cac7d94b0e0e467b577504c2f8a22c6


---
### PHẦN 4: NHẬN DIỆN NGÔN NGỮ BÀI ĐĂNG (LANGUAGE PROFILING)

Thị trường tuyển dụng IT Việt Nam có đặc thù phân mảnh ngôn ngữ:
- Công ty nội địa thường viết hoàn toàn bằng **Tiếng Việt (`vi`)**.
- Công ty đa quốc gia hoặc trung tâm phần mềm offshore thường viết bằng **Tiếng Anh (`en`)**.
- Nhiều doanh nghiệp sử dụng **Song ngữ (`vi-en`)**: phần giới thiệu công ty bằng tiếng Việt nhưng phần mô tả kỹ thuật lại viết bằng tiếng Anh.

#### Thuật toán nhận diện:
- Đếm tần suất ký tự có dấu tiếng Việt (`à, á, ả, ã, ạ, ă, â, đ, ê, ô, ơ, ư, ý...`).
- Đếm số từ khóa tiếng Việt đặc thù (`tuyển`, `công việc`, `yêu cầu`, `quyền lợi`, `kinh nghiệm`, `lương`).
- Đếm số từ khóa tiếng Anh đặc thù (`responsibilities`, `requirements`, `benefits`, `experience`, `skills`, `must have`).
- Phân loại: `vi` (Tiếng Việt thuần), `en` (Tiếng Anh thuần), `vi-en` (Song ngữ).


In [4]:
# Thực hiện nhận diện ngôn ngữ trên toàn bộ dữ liệu 2,752 tin
languages_detected = []

for idx, row in df.iterrows():
    combined_text = f"{str(row.get('description_raw', ''))} {str(row.get('requirements_raw', ''))}"
    lang = detect_language(combined_text)
    languages_detected.append(lang)

df['language_profile'] = languages_detected

# Thống kê tỷ lệ phân bố
lang_counts = df['language_profile'].value_counts()
lang_pct = (df['language_profile'].value_counts(normalize=True) * 100).round(2)

lang_summary_df = pd.DataFrame({
    "Ngôn ngữ": ["Tiếng Việt (vi)", "Tiếng Anh (en)", "Song ngữ (vi-en)"],
    "Mã": ["vi", "en", "vi-en"],
    "Số lượng tin": [lang_counts.get("vi", 0), lang_counts.get("en", 0), lang_counts.get("vi-en", 0)],
    "Tỷ lệ %": [lang_pct.get("vi", 0), lang_pct.get("en", 0), lang_pct.get("vi-en", 0)]
})

print("=== THỐNG KÊ PHÂN BỐ NGÔN NGỮ BÀI ĐĂNG TUYỂN DỤNG ===")
print(lang_summary_df.to_string(index=False))


=== THỐNG KÊ PHÂN BỐ NGÔN NGỮ BÀI ĐĂNG TUYỂN DỤNG ===
        Ngôn ngữ    Mã  Số lượng tin  Tỷ lệ %
 Tiếng Việt (vi)    vi          2002    72.75
  Tiếng Anh (en)    en           738    26.82
Song ngữ (vi-en) vi-en            12     0.44


---
### PHẦN 5: CHUẨN HÓA CHỨC DANH CÔNG VIỆC (JOB TITLE NORMALIZATION)

#### Vấn đề:
Nhà tuyển dụng thường lồng ghép thông tin lương thưởng, địa điểm, chế độ tuyển dụng vào tiêu đề:
- *Ví dụ thô*: `[Hà Nội] Senior Python Developer (Django / FastAPI) - Lương up to $2,500 - Đi làm ngay`
- Nếu gom nhóm theo chuỗi này, hệ thống sẽ coi mỗi tin là một vị trí khác nhau.

#### Kỹ thuật NLP áp dụng:
1. **Làm sạch tiền tố / hậu tố gây nhiễu**: Cắt bỏ các khối ngoặc vuông `[...]`, ngoặc đơn `(...)`, các cụm từ quảng cáo (`Lương up to...`, `Đi làm ngay`, `Thưởng ký HĐ`).
2. **Khai phá vị trí & ánh xạ Role Mapping**: Áp dụng hệ luật Regex quét qua các chức danh chuẩn: `Backend Developer`, `Frontend Developer`, `Fullstack Developer`, `Data Analyst`, `Data Engineer`, `DevOps Engineer`, `Mobile Developer`, `QA/QC Engineer`, `Solution Architect`...


In [5]:
# So sánh 10 ví dụ thực tế Before vs After khi chuẩn hóa chức danh
raw_titles_sample = [
    "[Hà Nội] Senior Python Backend Developer (Django/FastAPI) - Lương up to 40M",
    "Tuyển Dụng Kỹ Sư Cầu Nối (BrSE) Tiếng Nhật N2 - Remote",
    "Chuyên Viên Phân Tích Dữ Liệu (Data Analyst) - SQL / PowerBI",
    "Senior Java Software Engineer (Spring Boot, Microservices)",
    "Thực Tập Sinh Lập Trình Frontend (ReactJS / VueJS) - Hỗ trợ dấu mộc",
    "DevOps / SRE Engineer (Kubernetes, AWS Cloud, CI/CD Pipeline)",
    "Data Engineer - Xây dựng Data Pipeline & Data Warehouse (Spark / Airflow)",
    "QA/QC Automation Tester (Selenium / Python) - Đi làm ngay",
    "Full-Stack Web Developer (NodeJS / React / MongoDB)",
    "Mobile App Developer (Flutter / React Native) - Lương hấp dẫn"
]

normalized_results = []
for title in raw_titles_sample:
    norm_title = clean_job_title(title)
    normalized_results.append({
        "Tiêu đề thô (Before)": title,
        "Chức danh chuẩn hóa (After)": norm_title
    })

df_title_demo = pd.DataFrame(normalized_results)
print("=== KẾT QUẢ CHUẨN HÓA CHỨC DANH BẰNG NLP ===")
for idx, r in df_title_demo.iterrows():
    print(f"[{idx+1:02d}] {r['Tiêu đề thô (Before)']}")
    print(f"     -> Chuẩn hóa: {r['Chức danh chuẩn hóa (After)']}\n")


=== KẾT QUẢ CHUẨN HÓA CHỨC DANH BẰNG NLP ===
[01] [Hà Nội] Senior Python Backend Developer (Django/FastAPI) - Lương up to 40M
     -> Chuẩn hóa: Senior Python Backend Developer (Django/FastAPI)

[02] Tuyển Dụng Kỹ Sư Cầu Nối (BrSE) Tiếng Nhật N2 - Remote
     -> Chuẩn hóa: Kỹ Sư Cầu Nối (BrSE) Tiếng Nhật N2 - Remote

[03] Chuyên Viên Phân Tích Dữ Liệu (Data Analyst) - SQL / PowerBI
     -> Chuẩn hóa: Chuyên Viên Phân Tích Dữ Liệu (Data Analyst) - SQL / PowerBI

[04] Senior Java Software Engineer (Spring Boot, Microservices)
     -> Chuẩn hóa: Senior Java Software Engineer (Spring Boot, Microservices)

[05] Thực Tập Sinh Lập Trình Frontend (ReactJS / VueJS) - Hỗ trợ dấu mộc
     -> Chuẩn hóa: Thực Tập Sinh Lập Trình Frontend (ReactJS / VueJS) - Hỗ trợ dấu mộc

[06] DevOps / SRE Engineer (Kubernetes, AWS Cloud, CI/CD Pipeline)
     -> Chuẩn hóa: DevOps / SRE Engineer (Kubernetes, AWS Cloud, CI/CD Pipeline)

[07] Data Engineer - Xây dựng Data Pipeline & Data Warehouse (Spark / Airflow)
  

---
### PHẦN 6: BÓC TÁCH THỰC THỂ SỐ HỌC: MỨC LƯƠNG & SỐ NĂM KINH NGHIỆM

#### 1. Bóc tách mức lương (`parse_salary`):
- Trích xuất dải số từ các chuỗi như: `15 - 25 triệu`, `Up to 2,000 USD`, `30 Triệu`, `Thỏa thuận`.
- Xác định đơn vị tiền tệ (`VND` hoặc `USD`) và chu kỳ lương (`month` hoặc `year`).
- Quy đổi chuẩn hóa về **VNĐ/tháng** (tỷ giá tham chiếu USD/VND: 25,400).
- Phân loại định dạng lương: `range` (khoảng Min-Max), `max_only` (lên đến), `fixed` (lương cố định), `negotiable` (thỏa thuận).

#### 2. Bóc tách số năm kinh nghiệm (`parse_experience_years`):
- Trích xuất dải năm thâm niên: `2 - 3 năm`, `Trên 5 năm`, `1 năm`, `Không yêu cầu`.
- Gán phân lớp nhóm kinh nghiệm chuẩn hóa (`experience_bucket`):
  - `0 năm (Fresher/Intern)`
  - `1 năm`
  - `2 - 3 năm`
  - `3 - 5 năm`
  - `Trên 5 năm`


In [6]:
# Kiểm nghiệm bóc tách Lương và Kinh nghiệm trên các chuỗi văn bản phức tạp
salary_test_cases = [
    "15 - 25 triệu",
    "Up to 2,000 USD",
    "Từ 30.000.000 VNĐ",
    "Thoả thuận",
    "Lên tới 1,500$",
    "500 - 800 USD/tháng"
]

print("=== 1. KẾT QUẢ BÓC TÁCH THỰC THỂ MỨC LƯƠNG ===")
sal_demo = []
for s_raw in salary_test_cases:
    res = parse_salary(s_raw)
    sal_demo.append({
        "Lương thô": s_raw,
        "Định dạng": res["salary_type"],
        "Tiền tệ": res["salary_currency"],
        "Min (VNĐ)": f"{res['salary_min_vnd']:,.0f}" if res['salary_min_vnd'] else "None",
        "Max (VNĐ)": f"{res['salary_max_vnd']:,.0f}" if res['salary_max_vnd'] else "None"
    })
print(pd.DataFrame(sal_demo).to_string(index=False))

exp_test_cases = [
    "Không yêu cầu kinh nghiệm",
    "1 - 2 năm",
    "Tối thiểu 3 năm kinh nghiệm",
    "Trên 5 năm kinh nghiệm",
    "Fresher / Thực tập sinh",
    "2 - 4 năm kinh nghiệm làm việc"
]

def derive_bucket(min_y, max_y):
    if min_y == 0 and (max_y is None or max_y <= 1):
        return '0 năm (Fresher/Intern)'
    elif min_y == 1 or max_y == 1:
        return '1 năm'
    elif min_y is not None and 2 <= min_y <= 3:
        return '2 - 3 năm'
    elif min_y is not None and 3 < min_y <= 5:
        return '3 - 5 năm'
    elif min_y is not None and min_y > 5:
        return 'Trên 5 năm'
    return 'Không yêu cầu / Chưa rõ'

print("\n=== 2. KẾT QUẢ BÓC TÁCH THỰC THỂ KINH NGHIỆM ===")
exp_demo = []
for e_raw in exp_test_cases:
    min_y, max_y = parse_experience_years(e_raw)
    b_name = derive_bucket(min_y, max_y)
    exp_demo.append({
        "Kinh nghiệm thô": e_raw,
        "Số năm Min": min_y,
        "Số năm Max": max_y,
        "Nhóm chuẩn hóa (Bucket)": b_name
    })
print(pd.DataFrame(exp_demo).to_string(index=False))


=== 1. KẾT QUẢ BÓC TÁCH THỰC THỂ MỨC LƯƠNG ===
          Lương thô   Định dạng Tiền tệ  Min (VNĐ)  Max (VNĐ)
      15 - 25 triệu       range     VND 15,000,000 25,000,000
    Up to 2,000 USD    max_only     USD       None 50,800,000
  Từ 30.000.000 VNĐ undisclosed     VND       None       None
         Thoả thuận undisclosed     VND       None       None
     Lên tới 1,500$    max_only     USD       None 38,100,000
500 - 800 USD/tháng       range     USD 12,700,000 20,320,000

=== 2. KẾT QUẢ BÓC TÁCH THỰC THỂ KINH NGHIỆM ===
               Kinh nghiệm thô  Số năm Min  Số năm Max Nhóm chuẩn hóa (Bucket)
     Không yêu cầu kinh nghiệm         0.0         0.0  0 năm (Fresher/Intern)
                     1 - 2 năm         1.0         2.0                   1 năm
   Tối thiểu 3 năm kinh nghiệm         3.0         NaN               2 - 3 năm
        Trên 5 năm kinh nghiệm         5.0         NaN               3 - 5 năm
       Fresher / Thực tập sinh         NaN         NaN Không yêu cầu / Chư

---
### PHẦN 7: BÓC TÁCH KỸ NĂNG CÔNG NGHỆ CHUẨN HÓA (SKILL EXTRACTION / REGEX-NER)

#### 1. Taxonomy 76 Kỹ năng IT chuẩn hóa
Dự án quản lý từ điển chuẩn gồm 76 kỹ năng chia làm 10 nhóm chuyên môn:
1. `Programming Languages`: Python, Java, JavaScript, TypeScript, C++, C#, Golang, PHP, Ruby, Kotlin, Swift, Rust...
2. `Frontend Frameworks`: React, Vue.js, Angular, Next.js, Tailwind CSS, Redux...
3. `Backend Frameworks`: Node.js, Spring Boot, .NET, Django, FastAPI, Laravel, Express.js...
4. `Databases & Storage`: SQL, PostgreSQL, MySQL, MongoDB, Redis, Oracle, Elasticsearch...
5. `Cloud & DevOps`: Docker, Kubernetes, AWS, Azure, GCP, CI/CD, Linux, Terraform...
6. `Data Engineering & Big Data`: Spark, Kafka, Airflow, Hadoop, Hive, ETL...
7. `AI / Machine Learning / Data Science`: Machine Learning, Deep Learning, PyTorch, TensorFlow, NLP, Computer Vision, LLM, Pandas...
8. `Mobile Development`: Flutter, React Native, Android, iOS...
9. `Testing & QA`: Automation Testing, Selenium, Postman, JMeter, Unit Test...
10. `Architecture & Methodologies`: Microservices, RESTful API, GraphQL, Agile, Scrum, OOP...

#### 2. Kỹ thuật khử nhập nhằng từ vựng (Word Disambiguation):
- `Java` vs `JavaScript`: Dùng negative lookahead `\bjava\b(?!script)` để tránh bắt nhầm `JavaScript`.
- `C++` & `C#`: Xử lý ký tự đặc biệt `+` và `#` trong biểu thức chính quy.
- `.NET`: Sử dụng `(?:(?<!\w)\.net\b|\basp\.net\b|\bdotnet\b)` để không nhầm đuôi tên miền `.net`.
- `Golang`: Khử từ `go` tiếng Anh thông thường bằng cách chỉ nhận diện `golang` hoặc `go` đi kèm ngữ cảnh kỹ thuật (`go developer`, `go backend`...).
- `Computer Vision`: Khử từ viết tắt `CV` thông thường (Curriculum Vitae / Hồ sơ tuyển dụng).


In [7]:
# Chạy bộ matcher 76 kỹ năng trên 5 JD thực tế đại diện cho các lĩnh vực
skill_patterns = get_skill_patterns()

sample_jds = [
    {
        "role": "Python Data Engineer",
        "text": "Chúng tôi cần tuyển Data Engineer thành thạo Python, SQL và Spark. Kinh nghiệm xây dựng Data Pipeline trên AWS, sử dụng Kafka và Airflow. Biết Docker là điểm cộng."
    },
    {
        "role": "Senior Full-Stack Developer",
        "text": "Yêu cầu ứng viên thành thạo React, TypeScript và Next.js ở Frontend. Backend sử dụng Node.js hoặc NestJS, kết nối MongoDB và Redis. Hiểu biết CI/CD và Docker."
    },
    {
        "role": "Java Enterprise Backend",
        "text": "Phát triển hệ thống ngân hàng bằng Java (Spring Boot, Spring Framework). Làm việc với Oracle DB, PostgreSQL, kiến trúc Microservices và bảo mật RESTful API."
    },
    {
        "role": "AI / Machine Learning Engineer",
        "text": "Nghiên cứu mô hình Deep Learning sử dụng PyTorch hoặc TensorFlow. Kinh nghiệm xử lý bài toán NLP, Computer Vision và tích hợp mô hình LLM với kiến trúc RAG."
    },
    {
        "role": "Mobile App Engineer",
        "text": "Phát triển ứng dụng di động đa nền tảng bằng Flutter (Dart) hoặc React Native. Có kinh nghiệm với RESTful API, SQLite và Firebase."
    }
]

extracted_demo = []
for jd in sample_jds:
    found_skills = []
    text_lower = jd["text"].lower()
    for s_name, pattern in skill_patterns.items():
        if re.search(pattern, text_lower, re.IGNORECASE):
            found_skills.append(s_name)
    extracted_demo.append({
        "Vị trí": jd["role"],
        "Số kỹ năng bóc tách": len(found_skills),
        "Danh sách kỹ năng phát hiện": ", ".join(found_skills)
    })

print("=== KẾT QUẢ BÓC TÁCH KỸ NĂNG BẰNG REGEX-NER ===")
for r in extracted_demo:
    print(f"-> Vị trí: {r['Vị trí']}")
    print(f"   Kỹ năng trích xuất được ({r['Số kỹ năng bóc tách']} kỹ năng): {r['Danh sách kỹ năng phát hiện']}\n")


=== KẾT QUẢ BÓC TÁCH KỸ NĂNG BẰNG REGEX-NER ===
-> Vị trí: Python Data Engineer
   Kỹ năng trích xuất được (6 kỹ năng): Python, SQL, Docker, AWS, Spark, Kafka

-> Vị trí: Senior Full-Stack Developer
   Kỹ năng trích xuất được (10 kỹ năng): JavaScript, TypeScript, React, Next.js, Node.js, NestJS, MongoDB, Redis, Docker, CI/CD

-> Vị trí: Java Enterprise Backend
   Kỹ năng trích xuất được (5 kỹ năng): Java, Spring Boot, PostgreSQL, Oracle, Microservices

-> Vị trí: AI / Machine Learning Engineer
   Kỹ năng trích xuất được (6 kỹ năng): Deep Learning, PyTorch, TensorFlow, NLP, Computer Vision, LLM

-> Vị trí: Mobile App Engineer
   Kỹ năng trích xuất được (3 kỹ năng): Dart, React Native, Flutter



---
### PHẦN 8: PHÂN LOẠI TÍNH CHẤT YÊU CẦU: BẮT BUỘC VS ƯU TIÊN (REQUIREMENT CLASSIFICATION)

#### Phương pháp Section Context Parsing:
Bản mô tả công việc được chia làm 2 vùng ngữ cảnh chính:
1. **Vùng Bắt buộc (Required)**: Nằm dưới các tiêu đề:
   - `Yêu cầu công việc`, `Yêu cầu ứng viên`, `Must have`, `Kỹ năng bắt buộc`, `Minimum Qualifications`.
2. **Vùng Ưu tiên (Preferred / Nice-to-have)**: Nằm dưới các tiêu đề:
   - `Ưu tiên`, `Điểm cộng`, `Lợi thế`, `Nice to have`, `Preferred Qualifications`, `Plus`.

Nếu kỹ năng xuất hiện trong vùng ưu tiên, nó được gán nhãn `skill_type = 'preferred'`. Nếu xuất hiện trong vùng bắt buộc hoặc mô tả chung, nó được gán nhãn `skill_type = 'required'`.


In [8]:
# Mô phỏng bóc tách phân loại Required vs Preferred trên một JD mẫu
jd_with_sections = """
Yêu cầu bắt buộc:
- Có tối thiểu 2 năm kinh nghiệm lập trình Java và Spring Boot.
- Thành thạo truy vấn cơ sở dữ liệu SQL, MySQL hoặc PostgreSQL.
- Nắm vững kiến trúc RESTful API và Git.

Điểm cộng (Ưu tiên ứng viên):
- Có kinh nghiệm với Docker, Kubernetes (K8s).
- Biết thêm về AWS cloud hoặc Microservices là một lợi thế lớn.
"""

req_section_match = re.search(r'(?:yêu cầu|must have)[\s\S]*?(?=(?:điểm cộng|ưu tiên|nice to have|$))', jd_with_sections, re.IGNORECASE)
pref_section_match = re.search(r'(?:điểm cộng|ưu tiên|nice to have|lợi thế)[\s\S]*', jd_with_sections, re.IGNORECASE)

req_text = req_section_match.group(0).lower() if req_section_match else ""
pref_text = pref_section_match.group(0).lower() if pref_section_match else ""

req_skills = []
pref_skills = []

for s_name, pattern in skill_patterns.items():
    if pref_text and re.search(pattern, pref_text, re.IGNORECASE):
        pref_skills.append(s_name)
    elif req_text and re.search(pattern, req_text, re.IGNORECASE):
        req_skills.append(s_name)

print("=== KẾT QUẢ PHÂN LOẠI TÍNH CHẤT KỸ NĂNG ===")
print(f"1. Kỹ năng BẮT BUỘC (Required) : {', '.join(req_skills)}")
print(f"2. Kỹ năng ƯU TIÊN (Preferred) : {', '.join(pref_skills)}")


=== KẾT QUẢ PHÂN LOẠI TÍNH CHẤT KỸ NĂNG ===
1. Kỹ năng BẮT BUỘC (Required) : Java, Spring Boot, SQL, PostgreSQL, MySQL, Git
2. Kỹ năng ƯU TIÊN (Preferred) : Docker, Kubernetes, AWS, Microservices


---
### PHẦN 9: KHAI PHÁ MA TRẬN ĐỒNG XUẤT HIỆN (SKILL CO-OCCURRENCE MATRIX)

#### Ý nghĩa:
Ma trận đồng xuất hiện đo lường tần suất hai công nghệ được đòi hỏi cùng nhau trong một tin tuyển dụng.
- Giúp sinh viên và kỹ sư biết được "Combo công nghệ" thị trường đang tìm kiếm.
- Ví dụ: Người học `Python` thì nên trang bị kèm `SQL` và `FastAPI` (Backend) hay `Pandas` và `Spark` (Data).


In [9]:
# Tính toán mẫu ma trận đồng xuất hiện từ tập tin tuyển dụng thực tế
from collections import defaultdict
from itertools import combinations

# Lấy mẫu 300 tin để tính toán nhanh ma trận minh họa
cooccurrence_counts = defaultdict(int)
sample_subset = df.head(300)

for idx, row in sample_subset.iterrows():
    combined_desc = f"{str(row.get('description_raw', ''))} {str(row.get('requirements_raw', ''))}".lower()
    matched = [s for s, p in skill_patterns.items() if re.search(p, combined_desc, re.IGNORECASE)]
    # Đếm cặp xuất hiện đồng thời
    for s1, s2 in combinations(sorted(set(matched)), 2):
        cooccurrence_counts[(s1, s2)] += 1

# Sắp xếp top 15 cặp kỹ năng đi kèm nhau nhiều nhất
top_pairs = sorted(cooccurrence_counts.items(), key=lambda x: x[1], reverse=True)[:15]

pair_rows = []
for (s1, s2), cnt in top_pairs:
    pair_rows.append({"Kỹ năng 1": s1, "Kỹ năng 2": s2, "Số tin cùng đòi hỏi": cnt})

df_pairs = pd.DataFrame(pair_rows)
print("=== TOP 15 CẶP KỸ NĂNG CÔNG NGHỆ THƯỜNG ĐI ĐÔI NHẤT THỊ TRƯỜNG ===")
print(df_pairs.to_string(index=False))


=== TOP 15 CẶP KỸ NĂNG CÔNG NGHỆ THƯỜNG ĐI ĐÔI NHẤT THỊ TRƯỜNG ===
 Kỹ năng 1  Kỹ năng 2  Số tin cùng đòi hỏi
     CI/CD     Docker                   25
JavaScript TypeScript                   25
  HTML/CSS JavaScript                   23
JavaScript      React                   21
    Docker        Git                   21
JavaScript        SQL                   20
     React TypeScript                   19
       Git JavaScript                   19
PostgreSQL        SQL                   19
    Docker PostgreSQL                   19
       Git PostgreSQL                   19
       SQL SQL Server                   18
    Python        SQL                   17
       Git        SQL                   17
     CI/CD        Git                   17


---
### PHẦN 10: TỔNG KẾT - ÁNH XẠ DỮ LIỆU NLP VÀO DATABASE VÀ WEB DASHBOARD

Toàn bộ kết quả từ các bước xử lý NLP ở trên không chỉ dừng lại ở tính toán thử nghiệm mà được ánh xạ trực tiếp vào kiến trúc dữ liệu toàn diện của dự án:

#### 1. Ánh xạ vào các bảng PostgreSQL Silver Layer:
- **Bảng `jobs`**:
  - `title_normalized`: Lưu chức danh chuẩn hóa phục vụ gom nhóm phân loại.
  - `salary_min_vnd`, `salary_max_vnd`, `salary_type`: Lưu mức thu nhập đã bóc tách và quy đổi chuẩn về VNĐ.
  - `experience_min_years`, `experience_max_years`: Lưu dải số năm kinh nghiệm.
- **Bảng `job_descriptions`**:
  - `description_clean`, `requirements_clean`, `benefits_clean`: Văn bản đã loại bỏ mã HTML rác.
  - `language`: Ngôn ngữ nhận diện (`vi`, `en`, `vi-en`).
  - `text_hash`: Mã băm SHA-256 theo dõi biến động nội dung.
- **Bảng `job_skills`**:
  - Bảng quan hệ N-N lưu trữ từng cặp `(job_id, skill_id, skill_type)`.
  - `skill_type`: `required` (bắt buộc) hoặc `preferred` (ưu tiên).

#### 2. Ánh xạ vào các Analytical Views Gold Layer:
- `vw_gold_skill_demand`: Tổng hợp nhu cầu 76 kỹ năng, thị phần thâm nhập (`market_penetration_pct`), tỷ lệ bắt buộc so với ưu tiên.
- `vw_gold_skill_cooccurrence`: Ma trận đồng xuất hiện giữa các cặp công nghệ.
- `vw_gold_role_geo_matrix`: Ma trận nhu cầu theo chức danh chuẩn hóa và địa phương.
- `vw_gold_salary_experience_matrix`: Benchmark dải lương thực tế theo từng cấp bậc và số năm kinh nghiệm.
- `vw_gold_student_career`: Kho việc làm dành riêng cho sinh viên (Intern / Fresher / 0 năm kinh nghiệm).

#### 3. Trực quan hóa trên Streamlit Web Application:
- **Trang 1 - Thị Trường IT**: Biểu đồ cột ngang các chức danh hot nhất (`title_normalized`), hình thức làm việc và phân bố địa phương.
- **Trang 2 - Kỹ Năng & Yêu Cầu**: Biểu đồ Donut, Stacked Bar phân chia `Required vs Preferred`, bản đồ nhiệt `Heatmap` cho ma trận đồng xuất hiện và công cụ so sánh kỹ năng giữa 2 vai trò (`Data Analyst vs Data Engineer`).
- **Trang 3 - Lương & Kinh Nghiệm**: Biểu đồ dải lương theo thâm niên số năm kinh nghiệm và bảng xếp hạng các công nghệ đem lại thu nhập cao nhất.
- **Trang 4 - Gợi Ý Cho Sinh Viên**: Top kỹ năng cần học cho người mới bắt đầu và bảng tra cứu việc làm đầu vào đang mở nhận hồ sơ.
